# Fake News Classification with DistilBERT

Goal: fine-tune a small transformer (`distilbert-base-uncased`) on a short text classifier for "is this claim likely true or not". The data is the real LIAR dataset (Wang, 2017): about 12.8K short political statements (10,269 train rows) from PolitiFact, each labeled by a fact-checker. I download the original zip from UCSB. To keep it CPU-friendly I fine-tune on a random 2,000 statements from the official train split for 2 epochs and evaluate on the full official test split (1,283 rows in the file).

A few honest caveats up front:
- LIAR labels are 6-way (pants-fire / false / barely-true / half-true / mostly-true / true). I'm collapsing to binary (pants-fire, false, barely-true vs half-true, mostly-true, true) because 2,000 examples isn't enough to train 6 classes responsibly.
- Fake news detection at the *claim* level is a known hard problem. Published results on LIAR without metadata sit in the mid-20s for 6-way accuracy and the low-to-mid 60s for binary. I'm not going to beat that with 2,000 examples and 2 epochs.
- This is a learning project for the fine-tuning pipeline. I also fit a TF-IDF + logistic regression baseline on the same split so the DistilBERT number has something to stand next to.

If the HuggingFace model download fails (no internet, etc.), the notebook falls back to the TF-IDF baseline so there's still something to look at.

In [1]:
# !pip install transformers datasets scikit-learn torch -q
import warnings
warnings.filterwarnings("ignore")

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

random.seed(11)
np.random.seed(11)
torch.manual_seed(11)
torch.set_num_threads(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")


device: cpu


## Data: the LIAR dataset

In [2]:
import urllib.request, zipfile
from pathlib import Path

DATA = Path("../../data/liar")
ZIP = DATA / "liar_dataset.zip"
if not (DATA / "train.tsv").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://www.cs.ucsb.edu/~william/data/liar_dataset.zip", ZIP)
    zipfile.ZipFile(ZIP).extractall(DATA)

cols = ["id", "label6", "text", "subject", "speaker", "job", "state", "party",
        "barely_true_c", "false_c", "half_true_c", "mostly_true_c", "pants_fire_c", "context"]
def load(split):
    d = pd.read_csv(DATA / f"{split}.tsv", sep="\t", header=None, names=cols, quoting=3)
    true_ish = {"half-true", "mostly-true", "true"}
    d["label"] = d["label6"].isin(true_ish).astype(int)
    return d[["text", "label", "label6"]]

full_train = load("train")
test_df = load("test")
df = full_train.sample(n=2000, random_state=11).reset_index(drop=True)
print(f"LIAR train: {len(full_train)}  | sampled: {len(df)}  | test: {len(test_df)}")
print("label balance (sample):", df["label"].value_counts().to_dict())
df.head(3)

LIAR train: 10269  | sampled: 2000  | test: 1283
label balance (sample): {1: 1100, 0: 900}


,text,label,label6
0,If the U.S. didnt borrow $1.2 trillion every y...,0,false
1,Says there are $500 billion in Medicare cuts r...,0,barely-true
2,Says State Rep. Kristi Thibaut was an ACORN lo...,1,true


## Train / val split

I hold out 20% of the 2,000-row sample as a validation set and keep the official LIAR test split for the final number.

In [3]:
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=11, stratify=df["label"])
print(f"train: {len(train_df)}  val: {len(val_df)}")


train: 1600  val: 400


## Fine-tune DistilBERT (with a TF-IDF fallback if the HF model download fails)

In [4]:
USE_HF = True
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(device)
    print("HF model loaded:", model.config.model_type)
except Exception as e:
    USE_HF = False
    print("HF download / load failed — falling back to TF-IDF baseline.")
    print(repr(e))


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

... (2 outputs left out)

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


HF model loaded: distilbert


In [5]:
class ClaimsDS(Dataset):
    def __init__(self, texts, labels, tok, max_len=64):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tok = tok
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(self.texts[i], padding="max_length", truncation=True,
                       max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[i], dtype=torch.long),
        }

if USE_HF:
    train_ds = ClaimsDS(train_df["text"], train_df["label"], tokenizer)
    val_ds   = ClaimsDS(val_df["text"],   val_df["label"],   tokenizer)
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    EPOCHS = 2

    for ep in range(EPOCHS):
        model.train()
        total = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            out = model(input_ids=batch["input_ids"].to(device),
                        attention_mask=batch["attention_mask"].to(device),
                        labels=batch["label"].to(device))
            out.loss.backward()
            optimizer.step()
            total += out.loss.item()
        print(f"epoch {ep + 1}  avg train loss: {total / len(train_loader):.4f}")


epoch 1  avg train loss: 0.6878


epoch 2  avg train loss: 0.6179


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def bert_predict(texts):
    loader = DataLoader(ClaimsDS(texts, [0] * len(texts), tokenizer), batch_size=64, shuffle=False)
    out = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            logits = model(input_ids=batch["input_ids"].to(device),
                           attention_mask=batch["attention_mask"].to(device)).logits
            out.extend(logits.argmax(-1).cpu().tolist())
    return np.array(out)

# TF-IDF + logistic regression baseline on the same 1,600 training rows
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2).fit(train_df["text"])
lr = LogisticRegression(max_iter=1000, class_weight="balanced").fit(vec.transform(train_df["text"]), train_df["label"])

for split_name, d in [("validation (400 from train sample)", val_df), ("official LIAR test", test_df)]:
    print(f"=== {split_name} ===")
    p_lr = lr.predict(vec.transform(d["text"]))
    print(f"TF-IDF + LR  accuracy: {accuracy_score(d['label'], p_lr):.3f}  macro-F1: {f1_score(d['label'], p_lr, average='macro'):.3f}")
    if USE_HF:
        p_bert = bert_predict(d["text"])
        print(f"DistilBERT   accuracy: {accuracy_score(d['label'], p_bert):.3f}  macro-F1: {f1_score(d['label'], p_bert, average='macro'):.3f}")
    print()

if USE_HF:
    print(classification_report(test_df["label"], p_bert, target_names=["false-ish", "true-ish"], digits=3))

=== validation (400 from train sample) ===
TF-IDF + LR  accuracy: 0.570  macro-F1: 0.568


DistilBERT   accuracy: 0.588  macro-F1: 0.575

=== official LIAR test ===
TF-IDF + LR  accuracy: 0.592  macro-F1: 0.586


DistilBERT   accuracy: 0.595  macro-F1: 0.583

              precision    recall  f1-score   support

   false-ish      0.537     0.487     0.511       556
    true-ish      0.634     0.678     0.655       727

    accuracy                          0.595      1283
   macro avg      0.585     0.583     0.583      1283
weighted avg      0.592     0.595     0.593      1283



## Honest reading

- On the official LIAR test split (1,283 statements), 2 epochs of DistilBERT on 1,600 training statements gets **59.5% accuracy / 0.583 macro-F1**. A TF-IDF + logistic regression baseline on the same 1,600 rows gets 59.2% / 0.586. So the transformer is not buying anything yet at this data size; both sit right around the published binary LIAR numbers for text-only models.
- The train loss only dropped from 0.69 to 0.62 over 2 epochs, so the model is still close to the start of learning. The DistilBERT model is much more biased toward the "true-ish" class (recall 0.68 vs 0.49), which is partly just the 55/45 class balance in the sample.
- Claim-level fact checking from text alone is hard. Many LIAR statements are only checkable with outside knowledge, and the published gains on LIAR mostly come from metadata (speaker, party, the speaker's past "credit history" counts), which I deliberately left out.
- The real challenge in fake news detection is distribution shift: a model trained on 2007-2016 PolitiFact claims will not generalize to new topics. The right evaluation would split by time and topic, not at random.
- Labels come from fact-checkers, who themselves disagree, so any model trained on them inherits that noise.
- Next steps if I took this further: (1) train on all 10K statements for 3+ epochs on a GPU, (2) add the speaker metadata and credit-history counts, (3) compare DistilBERT, RoBERTa, and the TF-IDF baseline on a time-based split.
